# 01 — Image Quality Check

Purpose: Screen PoC photos so that captures unsuitable for landmark extraction can be rejected early.

Checks:
- Resolution (short side >= 480 px)
- Brightness (mean luma not at extremes)
- Sharpness (Laplacian variance)

Prerequisites:
- Place images under data/raw/front/ and data/raw/side/
- Kernel: it-signal-poc

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.image_quality import batch_assess

ROOT

In [ ]:
front_reports = batch_assess(ROOT / "data" / "raw" / "front")
side_reports = batch_assess(ROOT / "data" / "raw" / "side")

rows = []
for r in front_reports + side_reports:
    rows.append({
        "file": str(r.path),
        "w": r.width, "h": r.height,
        "brightness": round(r.brightness, 1),
        "sharpness": round(r.sharpness, 1),
        "passed": r.passed,
        "reasons": ", ".join(r.reasons),
    })
df = pd.DataFrame(rows)
df

In [ ]:
out = ROOT / "data" / "processed" / "quality_report.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f"wrote: {out}")